In [1]:
import onnx
import onnxruntime as ort
from torchvision import datasets, transforms
import numpy as np

from io import BytesIO
from urllib import request

from PIL import Image

In [2]:
def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img


def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

In [3]:
model = onnx.load("hair_classifier_v1.onnx")

graph = model.graph

print("Inputs:")
for inp in graph.input:
    print(" -", inp.name)

print("\nOutputs:")
for out in graph.output:
    print(" -", out.name)

Inputs:
 - input

Outputs:
 - output


In [4]:
url = "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"
raw_img = download_image(url)
img = prepare_image(raw_img, (200, 200))

In [5]:
train_transforms = transforms.Compose([
    transforms.Resize((200, 200)), 
    transforms.ToTensor(),           
    transforms.Normalize(        
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

img_tensor = train_transforms(img)

In [6]:
img_tensor

tensor([[[-1.0733, -1.0048, -1.0390,  ..., -1.0733, -1.0733, -1.2103],
         [-1.0562, -1.0219, -1.0048,  ..., -1.0219, -1.0562, -1.1760],
         [-0.9534, -0.9705, -0.9192,  ..., -1.0219, -1.1075, -1.2274],
         ...,
         [-1.6727, -1.6727, -1.5185,  ...,  1.7352,  1.6495,  1.7865],
         [-1.6727, -1.6555, -1.6384,  ...,  1.6838,  1.5982,  1.7009],
         [-1.6384, -1.7240, -1.6727,  ...,  1.7180,  1.7352,  1.6838]],

        [[-0.2150, -0.1099, -0.1099,  ..., -0.5126, -0.4776, -0.6352],
         [-0.1975, -0.1625, -0.1625,  ..., -0.5126, -0.5476, -0.6001],
         [-0.0749, -0.1275, -0.1450,  ..., -0.4776, -0.6001, -0.7052],
         ...,
         [-1.1429, -1.1429, -0.9678,  ...,  2.0609,  1.9734,  2.1134],
         [-1.1954, -1.1253, -1.0903,  ...,  2.0084,  1.9734,  2.0784],
         [-1.1954, -1.1954, -1.1429,  ...,  2.0434,  2.0784,  2.0609]],

        [[-1.4210, -1.2990, -1.2467,  ..., -1.8044, -1.7173, -1.7870],
         [-1.3513, -1.3164, -1.2641,  ..., -1

In [7]:
img_tensor = img_tensor.unsqueeze(0)
img_np = img_tensor.numpy().astype(np.float32)

In [8]:
session = ort.InferenceSession("hair_classifier_v1.onnx")

In [9]:
input_name = session.get_inputs()[0].name
print("Input name:", input_name)

outputs = session.run(None, {input_name: img_np})
print(outputs)

Input name: input
[array([[0.09156641]], dtype=float32)]
